# 06 MCTG: Multi-frequency Continuous-share Trading with GARCH

## 论文参考
- **作者**: Zhishun Wang
- **年份**: 2021
- **标题**: *MCTG: Multi-frequency continuous-share trading algorithm with GARCH*
- **摘要**: 本文提出MCTG算法，结合多频率特征（日频、5日、20日）与GARCH波动率建模，
  通过DQN强化学习智能体做出连续仓位决策。多频率特征融合使智能体同时感知短期动量、
  中期趋势和长期波动结构，GARCH模型提供前瞻性波动率估计作为额外状态信息。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### 1. 多频率特征构建
对原始日频价格序列，分别在三个时间尺度提取特征：
- **日频 (1d)**: 日收益率 $r_t$, 日内波幅 $(H_t - L_t)/C_t$, 成交量变化率
- **周频 (5d)**: $r_{t,5} = \frac{C_t - C_{t-5}}{C_{t-5}}$, 5日波动率 $\sigma_5$, 5日均线偏离
- **月频 (20d)**: $r_{t,20}$, 20日波动率 $\sigma_{20}$, 20日RSI

### 2. GARCH(1,1) 波动率建模
$$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$
其中 $\epsilon_t = r_t - \mu$ 为残差项。GARCH预测的条件波动率 $\hat{\sigma}_{t+1}$ 作为RL状态的额外维度。

### 3. DQN 强化学习
- **状态**: $s_t = [\text{multi-freq features}, \hat{\sigma}_{t+1}^{GARCH}, \text{position}]$
- **动作**: $a_t \in \{0 (空仓), 1 (半仓), 2 (满仓)\}$
- **奖励**: $r_t = a_t \cdot r_{t+1} - c \cdot |a_t - a_{t-1}|$ (收益减去换手成本)
- **Q网络**: $Q(s, a; \theta) \approx \mathbb{E}[R_t | s_t=s, a_t=a]$
- **损失**: $L(\theta) = \mathbb{E}[(r + \gamma \max_{a'} Q(s', a'; \theta^-) - Q(s, a; \theta))^2]$

In [ ]:
# === Cell 3: 多频率数据融合动画 ===
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)
T = 200
prices = 10 + np.cumsum(np.random.randn(T) * 0.1)
ret_1d = np.diff(prices) / prices[:-1]
ret_5d = np.array([prices[i] / prices[max(0, i-5)] - 1 for i in range(1, T)])
ret_20d = np.array([prices[i] / prices[max(0, i-20)] - 1 for i in range(1, T)])

frames = []
actions_demo = np.zeros(T - 1)
state_labels = []

for step in range(30, T - 1, 3):
    # Simulate DQN action based on multi-freq signal
    signal = 0.5 * ret_1d[step] + 0.3 * ret_5d[step] + 0.2 * ret_20d[step]
    if signal > 0.005:
        actions_demo[step] = 2  # full
    elif signal > -0.005:
        actions_demo[step] = 1  # half
    else:
        actions_demo[step] = 0  # empty

    buy_idx = [i for i in range(step) if actions_demo[i] == 2]
    half_idx = [i for i in range(step) if actions_demo[i] == 1]

    frame_data = [
        go.Scatter(x=list(range(step)), y=prices[:step], mode='lines',
                   name='价格', line=dict(color='gray', width=1.5)),
        go.Scatter(x=list(range(step)), y=ret_1d[:step] * 100, mode='lines',
                   name='日频收益(%)', line=dict(color='#2196F3', width=1), yaxis='y2'),
        go.Scatter(x=list(range(step)), y=ret_5d[:step] * 100, mode='lines',
                   name='5日收益(%)', line=dict(color='#FF9800', width=1), yaxis='y2'),
        go.Scatter(x=list(range(step)), y=ret_20d[:step] * 100, mode='lines',
                   name='20日收益(%)', line=dict(color='#9C27B0', width=1), yaxis='y2'),
    ]
    if buy_idx:
        frame_data.append(go.Scatter(
            x=buy_idx, y=[prices[i] for i in buy_idx], mode='markers',
            name='满仓', marker=dict(color='green', size=8, symbol='triangle-up')))
    if half_idx:
        frame_data.append(go.Scatter(
            x=half_idx, y=[prices[i] for i in half_idx], mode='markers',
            name='半仓', marker=dict(color='orange', size=6, symbol='diamond')))

    frames.append(go.Frame(data=frame_data, name=f'Step {step}'))

fig = go.Figure(data=frames[0].data if frames else [], frames=frames)
fig.update_layout(
    title=dict(text='MCTG: 多频率特征融合 -> RL状态 -> 交易决策'),
    xaxis_title='时间步', yaxis_title='价格',
    yaxis2=dict(title='收益率(%)', overlaying='y', side='right'),
    height=550, width=1000,
    updatemenus=[dict(type='buttons', buttons=[
        dict(label='▶ 播放', method='animate',
             args=[None, {'frame': {'duration': 200}}]),
    ])],
    sliders=[dict(
        steps=[dict(args=[[f.name]], label=f.name, method='animate') for f in frames],
    )],
)
fig.show()

In [ ]:
# === Cell 4: 导入与配置 ===
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque
import random

from shared.data_fetcher import get_stock_daily, get_index_daily
from shared.factors import rsi, macd, bollinger_bands, volatility, momentum
from shared.backtest_engine import Backtester
from shared.visualization import (plot_equity_curve, plot_drawdown,
                                   plot_metrics_table, plot_signals)
from shared.animation_helpers import animate_rl_trading
from shared.constants import *

print('All imports successful.')

In [ ]:
# === Cell 5: 数据获取 ===
stock = get_stock_daily(STOCK_MOUTAI, DEFAULT_START, DEFAULT_END)
benchmark = get_index_daily(CSI300_CODE, DEFAULT_START, DEFAULT_END)

close = stock['close']
high = stock['high']
low = stock['low']
volume = stock['volume']

print(f'Stock data: {len(stock)} trading days')
print(f'Date range: {stock.index[0]} ~ {stock.index[-1]}')

In [ ]:
# === Cell 6: 多频率特征工程 + GARCH ===

# --- 日频特征 ---
feat = pd.DataFrame(index=stock.index)
feat['ret_1d'] = close.pct_change(1)
feat['range_1d'] = (high - low) / close
feat['vol_chg'] = volume.pct_change(1)

# --- 5日 (周频) 特征 ---
feat['ret_5d'] = close.pct_change(5)
feat['vol_5d'] = feat['ret_1d'].rolling(5).std()
feat['ma5_dev'] = (close - close.rolling(5).mean()) / close.rolling(5).mean()

# --- 20日 (月频) 特征 ---
feat['ret_20d'] = close.pct_change(20)
feat['vol_20d'] = feat['ret_1d'].rolling(20).std()
feat['rsi_20'] = rsi(close, 20)

# --- GARCH(1,1) 条件波动率 ---
def simple_garch(returns, omega=1e-6, alpha=0.1, beta=0.85):
    """简化版GARCH(1,1)估计条件方差序列"""
    n = len(returns)
    sigma2 = np.zeros(n)
    sigma2[0] = returns.var()
    for t in range(1, n):
        sigma2[t] = omega + alpha * returns.iloc[t-1]**2 + beta * sigma2[t-1]
    return pd.Series(np.sqrt(sigma2), index=returns.index, name='garch_vol')

ret_series = feat['ret_1d'].fillna(0)
feat['garch_vol'] = simple_garch(ret_series)

# 归一化
feat_cols = ['ret_1d', 'range_1d', 'vol_chg', 'ret_5d', 'vol_5d', 'ma5_dev',
             'ret_20d', 'vol_20d', 'rsi_20', 'garch_vol']
feat.dropna(inplace=True)

for col in feat_cols:
    feat[col] = (feat[col] - feat[col].rolling(60).mean()) / (feat[col].rolling(60).std() + 1e-8)

feat.dropna(inplace=True)
print(f'Feature matrix shape: {feat.shape}')
print(f'Features: {feat_cols}')

In [ ]:
# === Cell 7: DQN 模型 ===

class ReplayBuffer:
    def __init__(self, capacity=5000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards),
                np.array(next_states), np.array(dones))

    def __len__(self):
        return len(self.buffer)


class SimpleQNetwork:
    """简单的表格+线性Q网络 (无需PyTorch依赖)"""
    def __init__(self, state_dim, n_actions, lr=0.001):
        self.state_dim = state_dim
        self.n_actions = n_actions
        self.lr = lr
        np.random.seed(42)
        # 两层线性网络权重
        self.W1 = np.random.randn(state_dim, 64) * 0.1
        self.b1 = np.zeros(64)
        self.W2 = np.random.randn(64, n_actions) * 0.1
        self.b2 = np.zeros(n_actions)

    def predict(self, state):
        """Forward pass: state -> Q values"""
        if state.ndim == 1:
            state = state.reshape(1, -1)
        h = np.maximum(0, state @ self.W1 + self.b1)  # ReLU
        q_values = h @ self.W2 + self.b2
        return q_values

    def update(self, states, actions, targets):
        """Simple gradient descent update"""
        h = np.maximum(0, states @ self.W1 + self.b1)
        q_values = h @ self.W2 + self.b2

        # Compute gradients for selected actions
        dq = np.zeros_like(q_values)
        for i, a in enumerate(actions):
            dq[i, a] = q_values[i, a] - targets[i]
        dq /= len(actions)

        # Backprop
        dW2 = h.T @ dq
        db2 = dq.sum(axis=0)
        dh = dq @ self.W2.T
        dh[h <= 0] = 0  # ReLU gradient
        dW1 = states.T @ dh
        db1 = dh.sum(axis=0)

        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def copy_from(self, other):
        self.W1 = other.W1.copy()
        self.b1 = other.b1.copy()
        self.W2 = other.W2.copy()
        self.b2 = other.b2.copy()


class DQNAgent:
    def __init__(self, state_dim, n_actions=3, lr=0.001, gamma=0.99,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995):
        self.n_actions = n_actions
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay

        self.q_network = SimpleQNetwork(state_dim, n_actions, lr)
        self.target_network = SimpleQNetwork(state_dim, n_actions, lr)
        self.target_network.copy_from(self.q_network)

        self.replay_buffer = ReplayBuffer(5000)
        self.batch_size = 64
        self.update_target_every = 50
        self.step_count = 0

    def select_action(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        q_values = self.q_network.predict(state)
        return np.argmax(q_values[0])

    def train_step(self):
        if len(self.replay_buffer) < self.batch_size:
            return 0.0

        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)

        # Target Q values
        next_q = self.target_network.predict(next_states)
        targets = rewards + self.gamma * np.max(next_q, axis=1) * (1 - dones)

        # Update Q network
        self.q_network.update(states, actions.astype(int), targets)

        self.step_count += 1
        if self.step_count % self.update_target_every == 0:
            self.target_network.copy_from(self.q_network)

        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

        # Return loss approximation
        current_q = self.q_network.predict(states)
        selected_q = np.array([current_q[i, a] for i, a in enumerate(actions.astype(int))])
        return np.mean((selected_q - targets) ** 2)

print('DQN agent defined.')

In [ ]:
# === Cell 8: 训练DQN ===

# Prepare data arrays
feature_matrix = feat[feat_cols].values
price_array = close.reindex(feat.index).values
returns_array = feat['ret_1d'].reindex(feat.index).fillna(0).values
n_days = len(feature_matrix)

# Split: 70% train, 30% test
train_end = int(n_days * 0.7)
state_dim = len(feat_cols) + 1  # features + current position

agent = DQNAgent(state_dim=state_dim, n_actions=3, lr=0.0005, gamma=0.95)
TRANSACTION_COST = 0.002

# Training loop
n_episodes = 15
episode_rewards = []
episode_losses = []

for ep in range(n_episodes):
    position = 0  # 0=empty, 1=half, 2=full
    total_reward = 0
    losses = []

    for t in range(train_end - 1):
        # Construct state: features + position
        state = np.append(feature_matrix[t], position / 2.0)
        action = agent.select_action(state)

        # Reward: position fraction * return - transaction cost
        position_frac = action / 2.0
        reward = position_frac * returns_array[t + 1]
        if action != position:
            reward -= TRANSACTION_COST * abs(action - position) / 2.0

        next_state = np.append(feature_matrix[t + 1], action / 2.0)
        done = (t == train_end - 2)

        agent.replay_buffer.push(state, action, reward, next_state, done)
        loss = agent.train_step()
        losses.append(loss)

        position = action
        total_reward += reward

    episode_rewards.append(total_reward)
    episode_losses.append(np.mean(losses))
    print(f'Episode {ep+1}/{n_episodes}: reward={total_reward:.4f}, '
          f'loss={np.mean(losses):.6f}, epsilon={agent.epsilon:.3f}')

print(f'\nTraining complete. Final epsilon: {agent.epsilon:.4f}')

In [ ]:
# === Cell 9: 生成交易信号并回测 ===

# Generate signals on FULL dataset (test starts at train_end)
all_actions = []
position = 0

for t in range(n_days):
    state = np.append(feature_matrix[t], position / 2.0)
    q_values = agent.q_network.predict(state)
    action = np.argmax(q_values[0])
    all_actions.append(action)
    position = action

# Map DQN actions to backtester signals: 0->0, 1->1, 2->1 (binary long/flat)
signals = pd.Series(
    [1 if a > 0 else 0 for a in all_actions],
    index=feat.index,
    name='signal'
)

# Backtest test period only
test_prices = close.reindex(feat.index).iloc[train_end:]
test_signals = signals.iloc[train_end:]
bench_close = benchmark['close'] if 'close' in benchmark.columns else None

bt = Backtester(initial_capital=INITIAL_CAPITAL, t_plus_1=True)
result = bt.run(test_prices, test_signals, bench_close)

print('=== MCTG DQN Backtest Results (Test Period) ===')
for k, v in result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# === Cell 10: 可视化 ===

# 1. Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(episode_rewards, 'b-o', markersize=4)
axes[0].set_title('训练累计奖励', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].grid(True, alpha=0.3)

axes[1].plot(episode_losses, 'r-o', markersize=4)
axes[1].set_title('训练损失', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Mean Loss')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 2. RL trading animation
test_actions = all_actions[train_end:]
test_rewards_anim = returns_array[train_end:]
fig_anim = animate_rl_trading(
    test_prices.values, test_actions, test_rewards_anim,
    title='MCTG DQN 交易过程动画'
)
fig_anim.show()

# 3. Equity curve
bench_eq = None
if result.get('benchmark_returns') is not None:
    bench_eq = INITIAL_CAPITAL * (1 + result['benchmark_returns']).cumprod()
plot_equity_curve(result['equity_curve'], bench_eq,
                  title='MCTG DQN - 累计收益 vs 沪深300')

# 4. Drawdown
plot_drawdown(result['equity_curve'], title='MCTG DQN - 回撤')

# 5. Metrics table
plot_metrics_table(result['metrics'], title='MCTG DQN - 绩效指标')

## 结果讨论

### 核心发现
1. **多频率特征融合**: 日频捕捉短期动量信号，周频/月频提供趋势确认，GARCH波动率帮助识别高波动环境
2. **DQN决策**: 智能体学会在低波动上升趋势中保持满仓，在高波动或下行环境中减仓或空仓
3. **GARCH贡献**: 条件波动率预测使智能体能够预判风险放大阶段，提前降低仓位

### 与论文对比
- Wang (2021) 使用连续动作空间和完整GARCH参数估计
- 本实现简化为离散动作 (空仓/半仓/满仓) 和固定GARCH参数
- 核心思想一致：多频率信息融合 + 波动率感知 + RL决策

### 改进方向
- 使用ARCH库进行完整GARCH参数MLE估计
- 扩展为连续动作空间 (DDPG/PPO)
- 增加更多频率维度（分钟级、小时级）